# 🏛️ Fine-Tuning vilm/vinallama-7b-chat trên Google Colab T4 Free
### Dự án: Trợ lý AI Tư vấn Pháp luật An toàn Giao thông đường bộ Việt Nam

Notebook chuẩn hóa này được thiết kế tối ưu hóa 100% cho phần cứng **GPU Tesla T4 (Free Tier trên Colab)**:
- **Mô hình nền (Base Model)**: `vilm/vinallama-7b-chat` (LLaMA-2 7B tinh chỉnh tiếng Việt).
- **Tập dữ liệu huấn luyện**: 683 cặp câu hỏi - đáp pháp luật giao thông đã được chuẩn hóa theo Luật TTATGTĐB 2024 & Nghị định 168/2024/NĐ-CP.
- **Kỹ thuật huấn luyện**: **4-bit QLoRA** (tiết kiệm 70% VRAM, tiêu hao ~5.5GB / 15GB VRAM của T4).
- **Chế độ kiểm định (Validation)**: Theo dõi `eval_loss` sau mỗi epoch để chống học vẹt (overfitting).
- **Đóng gói xuất xưởng**: Hợp nhất trọng số và Lượng tử hóa sang **GGUF (Q4_K_M)** mang tên **`vinallama-7b-chat-traffic-law.Q4_K_M.gguf`**.

## Bước 1: Kiểm tra phần cứng GPU & Cài đặt môi trường chuẩn
Lưu ý: Hãy đảm bảo bạn đã chọn **Runtime -> Change runtime type -> T4 GPU** trước khi chạy.

In [ ]:
# 1. Kiểm tra thông số GPU
!nvidia-smi

# 2. Cài đặt các thư viện cần thiết cho QLoRA Fine-tuning
!pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.46.1 datasets peft==0.13.2 trl==0.12.0 bitsandbytes accelerate

# 3. Gỡ bỏ gói torchao cũ của Colab để triệt tiêu lỗi xung đột phiên bản với peft
!pip uninstall -y -q torchao
print("\n>>> Đã thiết lập môi trường hoàn tất sạch sẽ!")


## Bước 2: Nạp dữ liệu huấn luyện (`train.jsonl` và `val.jsonl`)
Nếu bạn đã kéo thả 2 file vào thanh công cụ Files bên trái, ô này sẽ tự động nhận diện và nạp vào hệ thống.

In [ ]:
import os
from google.colab import files

if not os.path.exists('train.jsonl') or not os.path.exists('val.jsonl'):
    print("Chưa thấy file trong thư mục. Hãy chọn tải lên 2 file 'train.jsonl' và 'val.jsonl':")
    files.upload()

assert os.path.exists('train.jsonl'), 'Thiếu file train.jsonl!'
assert os.path.exists('val.jsonl'), 'Thiếu file val.jsonl!'
print(">>> Đã nhận đủ dữ liệu train.jsonl và val.jsonl thành công!")


## Bước 3: Đọc dữ liệu & Kiểm tra cấu trúc phân tầng

In [ ]:
import json
from datasets import Dataset

def load_jsonl_to_dataset(file_path):
    conversations = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                conversations.append(json.loads(line))
    return Dataset.from_list(conversations)

train_dataset = load_jsonl_to_dataset('train.jsonl')
val_dataset = load_jsonl_to_dataset('val.jsonl')

print(f"Số lượng mẫu tập Huấn luyện (Train): {len(train_dataset)} mẫu (80%)")
print(f"Số lượng mẫu tập Kiểm định (Val)   : {len(val_dataset)} mẫu (10%)")
print("\nMẫu đầu tiên tập Train:")
print("System Prompt:", train_dataset[0]['messages'][0]['content'])
print("Question     :", train_dataset[0]['messages'][1]['content'][:120], "...")


## Bước 4: Tải Base Model (`vilm/vinallama-7b-chat`) ở chế độ 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "vilm/vinallama-7b-chat"

# Cấu hình 4-bit Quantization chuẩn cho GPU T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # T4 dùng float16
    bnb_4bit_use_double_quant=True
)

print("Đang tải Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Đang tải Base Model vào GPU (mất khoảng 2-3 phút)...\n")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False
print(">>> Tải mô hình thành công! Bộ nhớ VRAM đang sử dụng:")
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv


## Bước 5: Cấu hình LoRA Adapters (PEFT)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,                         # LoRA rank
    lora_alpha=32,                # Hệ số co giãn trọng số
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## Bước 6: Huấn luyện Mô hình với SFTTrainer (3 Epochs)
- Đã bật `gradient_checkpointing_kwargs={'use_reentrant': False}` để dập tắt hoàn toàn UserWarning của PyTorch.
- Đã cấu hình lưu checkpoint tự động và theo dõi `eval_loss`.

In [ ]:
import shutil
from transformers import TrainingArguments
from trl import SFTTrainer

def format_prompts_fn(example):
    output_texts = []
    for messages in example['messages']:
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        output_texts.append(text)
    return output_texts

training_args = TrainingArguments(
    output_dir="./traffic_law_checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,         # Effective batch size = 2 * 4 = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False}, # Triệt tiêu PyTorch UserWarning
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=peft_config,
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
    formatting_func=format_prompts_fn
)

print("Bắt đầu huấn luyện mô hình...")
trainer.train()

# Lưu thư mục Adapter LoRA
adapter_dir = "./vinallama_traffic_law_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

# Nén sẵn file zip dự phòng
shutil.make_archive("vinallama_traffic_law_adapter", "zip", adapter_dir)
print("\n>>> Huấn luyện hoàn tất! Đã lưu Adapter và nén file vinallama_traffic_law_adapter.zip an toàn!")


## Bước 7: Kiểm tra suy luận thử nghiệm (Inference Test)
Đánh giá ngay khả năng viện dẫn Luật TTATGTĐB 2024 và Nghị định 168/2024/NĐ-CP của mô hình vừa fine-tune.

In [ ]:
test_question = "Người điều khiển xe ô tô lùi xe trên đường cao tốc bị xử phạt như thế nào theo Nghị định 168/2024/NĐ-CP?"

messages = [
    {"role": "system", "content": "Bạn là Trợ lý AI Tư vấn Pháp luật An toàn Giao thông đường bộ tại Việt Nam. Hãy trả lời câu hỏi của người dùng một cách chính xác, đúng căn cứ pháp lý hiện hành, rõ ràng và chuẩn mực."},
    {"role": "user", "content": test_question}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.2,
        repetition_penalty=1.1
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=== CÂU HỎI THỬ NGHIỆM ===\n", test_question)
print("\n=== CÂU TRẢ LỜI CỦA MÔ HÌNH VỪA FINE-TUNE ===\n", response)


## Bước 8: Hợp nhất (Merge) LoRA Adapter vào Mô hình gốc
- Thiết lập `device_map={"": 0}` để GPU T4 chứa trọn vẹn cả 32 tầng trên VRAM.
- Không offload sang CPU/meta device, dập tắt hoàn toàn các cảnh báo sao chép.

In [ ]:
import gc
from peft import PeftModel

# 1. Thu hồi toàn bộ bộ nhớ cũ
del model
del trainer
gc.collect()
torch.cuda.empty_cache()

adapter_path = "./vinallama_traffic_law_adapter"
merged_path = "./vinallama_traffic_law_merged"

print("Đang nạp Base Model vào GPU VRAM (ép trọn vẹn 32 tầng trên GPU 0)...\n")
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map={"": 0},              # Nạp 100% lên GPU số 0, không dùng meta device
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)

print("Đang hợp nhất LoRA Adapter vào mô hình gốc...")
merged_model = PeftModel.from_pretrained(base_model, adapter_path)
merged_model = merged_model.merge_and_unload()

print("Đang lưu mô hình hoàn chỉnh ra đĩa...")
merged_model.save_pretrained(merged_path, max_shard_size="4GB")
tokenizer.save_pretrained(merged_path)

# Giải phóng VRAM để nhường chỗ cho khâu Quantize
del base_model
del merged_model
gc.collect()
torch.cuda.empty_cache()

print(">>> MERGE HOÀN TẤT THÀNH CÔNG 100% CẢ 32 TẦNG!")


## Bước 9: Chuyển đổi sang GGUF (Q4_K_M) bằng `llama.cpp`
Tạo file định danh chuẩn xác cho đồ án: **`vinallama-7b-chat-traffic-law.Q4_K_M.gguf`**.

In [ ]:
# 1. Clone và biên dịch llama.cpp
!git clone https://github.com/ggerganov/llama.cpp.git
!cd llama.cpp && make -j llama-quantize
!pip install -q -r llama.cpp/requirements.txt

# 2. Convert từ định dạng HuggingFace sang GGUF FP16
!python llama.cpp/convert_hf_to_gguf.py ./vinallama_traffic_law_merged --outfile ./vinallama_traffic_law_fp16.gguf

# 3. Quantize sang Q4_K_M
!./llama.cpp/llama-quantize ./vinallama_traffic_law_fp16.gguf ./vinallama-7b-chat-traffic-law.Q4_K_M.gguf Q4_K_M

# 4. Kiểm tra kích thước file mô hình đầu ra (~4.1GB)
!ls -lh vinallama-7b-chat-traffic-law.Q4_K_M.gguf


## Bước 10: Tự động sao lưu Mô hình & Adapter vào Google Drive
Lưu trữ vĩnh viễn cả file GGUF lẫn file Adapter zip vào Google Drive của bạn.

In [ ]:
from google.colab import drive
import shutil

print("Đang kết nối với Google Drive...")
drive.mount('/content/drive')

# Sao chép file GGUF sang Drive
gguf_dest = '/content/drive/MyDrive/vinallama-7b-chat-traffic-law.Q4_K_M.gguf'
shutil.copy('vinallama-7b-chat-traffic-law.Q4_K_M.gguf', gguf_dest)
print(f"1. Đã lưu thành công file GGUF vào: {gguf_dest}")

# Sao chép file Adapter zip dự phòng sang Drive
adapter_dest = '/content/drive/MyDrive/vinallama_traffic_law_adapter.zip'
shutil.copy('vinallama_traffic_law_adapter.zip', adapter_dest)
print(f"2. Đã lưu thành công file Adapter zip vào: {adapter_dest}")

print("\n🎉 CHÚC MỪNG BẠN! TOÀN BỘ TIẾN TRÌNH FINE-TUNE ĐÃ HOÀN TẤT XUẤT SẮC!")
